In [1]:
import kagglehub
import os

path = kagglehub.dataset_download("andrewmvd/steam-reviews")
print("Path to dataset files:", path)
print(os.listdir(path))

c:\Users\ajdab\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\ajdab\.cache\kagglehub\datasets\andrewmvd\steam-reviews\versions\3
['dataset.csv']


Importing the Data set. Our data set is a set of steam reviews.

It contains: 
Game ID, Game Name, Review Text, Review Score, and Review vote
Note: large data set, about 6.4M reviews and 2.16GB of space

Something to note in general: The actual data has about 5.26M Postive rows 1.16M Negative rows which roughly 82% positive so just take into account.

In [2]:
import pandas as pd

csv_path = os.path.join(path, "dataset.csv")   

cols = ["app_id", "app_name", "review_text", "review_score", "review_votes"]
df = pd.read_csv(csv_path, usecols=cols)

Firstly 6.4M rows is far to great to work with.

Want to take a subsample of rows and ensure its random as data is sorted by game.

In [47]:
seed = 42

sample_df = df.sample(n = 100000, random_state = seed) #make sure we use a seed for reproducibility


Next we will clean our data checking for duplicates, nulls, and empty strings, additionally wanted to look at review lenght.

In [48]:
cleaned_df = sample_df.copy()

# Replace missing reviews
if cleaned_df["review_text"].isnull().sum() > 0:
    cleaned_df["review_text"] = cleaned_df["review_text"].fillna("No review text")

# Remove empty reviews
cleaned_df = cleaned_df[cleaned_df["review_text"].str.strip() != ""]

# Remove duplicate reviews
cleaned_df = cleaned_df.drop_duplicates(subset=["review_text"])

# Calculate review length
cleaned_df["review_length"] = cleaned_df["review_text"].str.len()

download sklearn if not installed.

In [24]:
pip install -U scikit-learn

   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   --------------------------- ------------ 5.8/8.4 MB 30.1 MB/s eta 0:00:01
   ---------------------------------------- 8.4/8.4 MB 24.7 MB/s  0:00:00
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   ------- -------------------------------- 7.3/37.4 MB 34.7 MB/s eta 0:00:01
   --------------- ------------------------ 14.7/37.4 MB 34.6 MB/s eta 0:00:01
   ----------------------- ---------------- 21.5/37.4 MB 33.3 MB/s eta 0:00:01
   ------------------------------ --------- 28.0/37.4 MB 32.7 MB/s eta 0:00:01
   ------------------------------------- -- 34.6/37.4 MB 32.5 MB/s eta 0:00:01
   ---------------------------------------- 37.4/37.4 MB 30.1 MB/s  0:00:01

   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ -----


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


import sklearn and set up our variables.

Additionally split our data sample into training and test data 80% and 20%.

In [ ]:
from sklearn import model_selection

x = cleaned_df["review_text"]
y = cleaned_df["review_score"]

#Split into training 80% and testing 20%
x_train, x_test, y_train, y_test = model_selection.train_test_split(x, y, test_size = 0.2, random_state = seed, stratify = y)

Vectorize our review text, so we it can be read.

In [50]:
fit_transformer = sk.feature_extraction.text.TfidfVectorizer()


x_train_tfidf = fit_transformer.fit_transform(x_train)
x_test_tfidf = fit_transformer.transform(x_test)

In [51]:
print(x_train_tfidf.shape)
print(x_test_tfidf.shape)

(65820, 67071)
(16456, 67071)


Now check and make sure we are close to 82% 18% split. 

In [42]:
print(pd.Series(y_train).value_counts(normalize=True))
print(pd.Series(y_test).value_counts(normalize=True))

review_score
 1    0.82443
-1    0.17557
Name: proportion, dtype: float64
review_score
 1    0.824441
-1    0.175559
Name: proportion, dtype: float64


Get a basline model and predictions set up, will be used for scoring later.

In [ ]:
import sklearn.dummy as dm#imports needed for a baseline model
from sklearn import metrics

baseline_model = dm.DummyClassifier(strategy = "most_frequent")
baseline_model.fit(x_train_tfidf, y_train) #learns the most frequent class from the training data

baseline_predictions =baseline_model.predict(x_test_tfidf)

Score the baseline prediction against the testing(y_test).

In [53]:
baseline_accuracy = sklearn.metrics.accuracy_score(y_test, baseline_predictions)
baseline_confusion_matrix = sklearn.metrics.confusion_matrix(y_test, baseline_predictions)
baseline_classification_report = sklearn.metrics.classification_report(y_test, baseline_predictions)

print("Baseline Model Accuracy:", baseline_accuracy)
print("Baseline Model Confusion Matrix:\n", baseline_confusion_matrix)
print("Baseline Model Classification Report:\n", baseline_classification_report)


Baseline Model Accuracy: 0.8244409333981526
Baseline Model Confusion Matrix:
 [[    0  2889]
 [    0 13567]]
Baseline Model Classification Report:
               precision    recall  f1-score   support

          -1       0.00      0.00      0.00      2889
           1       0.82      1.00      0.90     13567

    accuracy                           0.82     16456
   macro avg       0.41      0.50      0.45     16456
weighted avg       0.68      0.82      0.75     16456



c:\Users\ajdab\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ajdab\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ajdab\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

After setting up our baseline we want to set up our different models, starting with linear regression.

In [ ]:
from sklearn import linear_model as lm

#To options here that I can use, balanced or None. There is a tradeoff between these two that I will mention later.
lr_model = lm.LogisticRegression(max_iter = 1000, class_weight = "balanced", random_state = seed)

lr_model.fit(x_train_tfidf, y_train)
lr_predictions = lr_model.predict(x_test_tfidf)

#Score our model the same way we scored baseline
lr_accuracy = sklearn.metrics.accuracy_score(y_test, lr_predictions)
lr_confusion_matrix = sklearn.metrics.confusion_matrix(y_test, lr_predictions)
lr_classification_report = sklearn.metrics.classification_report(y_test, lr_predictions)


print("Logistic Regression Model Accuracy:", lr_accuracy)
print("Logistic Regression Model Confusion Matrix:\n", lr_confusion_matrix)
print("Logistic Regression Model Classification Report:\n", lr_classification_report)

Logistic Regression Model Accuracy: 0.8949927078269324
Logistic Regression Model Confusion Matrix:
 [[ 1544  1345]
 [  383 13184]]
Logistic Regression Model Classification Report:
               precision    recall  f1-score   support

          -1       0.80      0.53      0.64      2889
           1       0.91      0.97      0.94     13567

    accuracy                           0.89     16456
   macro avg       0.85      0.75      0.79     16456
weighted avg       0.89      0.89      0.89     16456



Testing our models precision for negative cases and the amount of false negatives. 

In [61]:
from sklearn.metrics import precision_score, recall_score, f1_score

lr_neg_precision = precision_score(y_test, lr_predictions, pos_label=-1)
lr_neg_recall    = recall_score(y_test, lr_predictions, pos_label=-1)
lr_neg_f1        = f1_score(y_test, lr_predictions, pos_label=-1)

print("Negative class precision:", lr_neg_precision)
print("Negative class recall:   ", lr_neg_recall)
print("Negative class F1:       ", lr_neg_f1)

Negative class precision: 0.8012454592631033
Negative class recall:    0.5344409830391139
Negative class F1:        0.6411960132890365


After testing balanced or not has a trade-off for classweight. 

Balanced classweight catches more negative reviews and misses less negative reviews. But it wrongly flags more postive reviews as negative.

On the otherhand unweighted catches less reviews, but has less false negatives.

In total 2889 negative reviews.

Unweighted:
Catches: about 1540
Misses: about 1350
false negatives: 380

Balanced:
Catches: about 2410480
Misses: about 480
false negatives: 1830

This happens due to the nature of the data set, 82% positive and 18% negative. 

The model learns positive is the safe anwser therefore with unweighted weight, the borderline negative reviews slip through the cracks. 

Decided on balanced weights as the main goal is to find the negative reviews and the common theme among them. Therefor missing negatives is much more costly then having false negatives.

Now we will work on a tree-based model. Note this will be more difficult due to the features and the amount of the features.

In [62]:
from sklearn.feature_extraction.text import TfidfVectorizer

#Want to limit features to 10K and ignore words that appear in less than 5 reviews.
#Additionally, only use unigrams single words, and not pairs.
tfidf_cap = TfidfVectorizer(max_features=10000, min_df=5, ngram_range=(1, 1))
x_train_cap = tfidf_cap.fit_transform(x_train)   # learn vocabulary from TRAIN only
x_test_cap = tfidf_cap.transform(x_test)         # apply it to test
print(x_train_cap.shape, x_test_cap.shape)

(65820, 10000) (16456, 10000)


Set up our baseline and logisticRegression model the same as before but with the limited features to make comparsion fair. 

In [64]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

baseline = DummyClassifier(strategy="most_frequent").fit(x_train_cap, y_train)
lr_plain = LogisticRegression(max_iter=1000, random_state=seed).fit(x_train_cap, y_train)
lr_bal = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=seed).fit(x_train_cap, y_train)

our tree-based model random forest with the same limited feautures.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=seed,
)
rf_model.fit(x_train_cap, y_train)